# Date grouping revisited

This notebook describes how the date grouping works in bps_to_omop.

## 1.1 Remove overlap

### General rationale

Imagine this as a visit in a timeline

```python
-(- - -)- - - - - - - -
 |     |
 |     |-> End of the visit
 |
 |-> Start of the visit
```
We want to find the **main visits**:

- A main visit contains other visits. 
- Main visits can be next to each other, ie: The end of the 1st can be the start of the 2nd
- Main visits can not overlap each other, ie: The end of the 1st can not be after the start of the 2nd
- Only main visits can populate the VISIT_OCCURRENCE table

If we sort the visits by person_id (asc), start_date (asc), end_date (desc) y type_concet (asc), we will have something like this for each person:
```python
-(- - -)- - - - - - - - 
-(- -)- - - - - - - - -
-(-)- - - - - - - - - -
- -(- - -)- - - - - - -
- -(- -)- - - - - - - -
- -(-)- - - - - - - - - 
- - -(- - -)- - - - - -
- - -(- -)- - - - - - -
- - -(-)- - - - - - - -
- - - - - -(- - -)- - - 
- - - - - -(- -)- - - -
- - - - - -(-)- - - - -
```

If we compare each visit with the **FIRST ONE**, there are different cases here:
1. **COMPLETELY CONTAINED VISITS**: Contained visits are completely contained in the first visit we are considering (This includes consecutive single day visits)
2. **PARTIALLY CONTAINED VISITS**:The start of the visit happens after the start of the 1st, but end of the happens after the end of the 1st
    - Starts afterwards, but extends further into the future.
3. **NOT CONTAINED VISITS**: The start of the visit is after the 1st. It is a "new" main visit.

```python
-(- - -)- - - - - - - - # This is a main visit
-(- -)- - - - - - - - - 1. contained
-(-)- - - - - - - - - - 1. contained
- -(- - -)- - - - - - - 2. partial
- -(- -)- - - - - - - - 1. contained
- -(-)- - - - - - - - - 1. contained
- - -(- - -)- - - - - - 2. partial
- - -(- -)- - - - - - - 2. partial
- - -(-)- - - - - - - - 1. contained
- - - - - -(- - -)- - - # This is the next main visit
- - - - - -(- -)- - - - 1. contained
- - - - - -(-)- - - - - 1. contained
```

Completely contained visits are the easy ones. We can link them to the main visit and remove them.

Partially contained visits are problematic. These will force us to extend our initial visit further into the future, essentially creating a new record.




### Pseudocode explanation

First we join together all files to be used. This essentialy is the VISIT_DETAIL table.

With this initial table we do the following:

1. Build the frame for the VISIT_DETAIL table (`build_visit_detail()`).
   1. Sort by `person_id`(asc), `start_date` (asc), `end_date` (desc) and `type_concept` (asc).
      * This way, we make sure that, for each `start_date`, the first column will have the latest `end_date`. This latest `end_date` is the visit that might contain other visits with the same `start_date`.
      * The `type_concept` column should be transformed first to a category datatype with a predefined order. This way the sorting will take that into account as well.
   2. Generate the field `visit_detail_id` with every visit.
   3. Rename columns to fit the VISIT_DETAIL structure.
   
With this, we have the VISIT_DETAIL table. From here we can build VISIT_OCCURRENCE on top of VISIT_DETAIL.

2. Extend the VISIT_DETAIL table to prepare for VISIT_OCCURRENCE generation (`build_visit_detail_extended()`):
   1. Generate placeholders for future VISIT_OCCURRENCE columns and for temporary columns.
      1. Group by each patient, generating:
        - **visit_start_datetime**: Initially the first **visit_detail_start_datetime**. It will be the **visit_start_datetime** VISIT_OCCCURRENCE column.
        - **visit_end_datetime**: Initially the first **visit_detail_end_datetime**. It will be the **visit_start_datetime** VISIT_OCCCURRENCE column. 
          - If visits needs to be joined together, this record will have the latest joined visit.
        - **visit_detail_id_original**: Initially the first **visit_detail_id**. It will be the **visit_detail_id** VISIT_OCCCURRENCE column.
        - **main_visit**: Initially will have a constant *"Unknown"* value for every record. It will be updated to identify main visits (Changed to "Yes") and the rest (Changed to "No").
      2. Join with VISIT_DETAIL to get the first row (earliest start_date, latest end_date) for each patient.
   2. Generate the *main_visit* column.
      - This field has 4 possibilities: 
         - *Yes:* Visit is a **main_visit**.
         - *No:* It was checked and visit is NOT a **main_visit**.
         - *Unknown:* We do not know if it is **main_visit** yet.
       - We create this columns as a category. Initially all records are *Unknown*. 
   3. Generate the auxiliary columns:
      - `is_contained`, to mark completely contained records.
      - `is_partial`, to mark partialy contained records.
      - `is_not_contained`, to marked not contained records, which could be main visits.
      - `parent_visit_detail_id`, to record the associated **main_visit** when it is found.

The previous operations only need to be done once at the start.

Afterwards, we will need to do the following operations in a loop until no *Unknown* visits are left:

3. Identify the next batch of **main_visit** (`identify_next_main_visits()`):
   1. They are the most recent that verify:
      1. They are *"Unknown"*
      2. **visit_detail_id** == **visit_detail_id_original**
   2. Promote **main_visit** value to *"Yes"*
4. Check how the other visits are related to the main visit
   1. Check if they are completely contained (`identify_contained_rows()`):
      1. If these conditions are verified:
         - **main_visit** -> *"Unknown"*
         - **visit_start_datetime** <= **"visit_detail_start_datetime"**
         - **visit_end_datetime** >= **"visit_detail_end_datetime"**
      2. Promote **is_contained** value to *True*
   2. For contained visits do (`update_contained_rows()`):
      1. Promote **main_visit** value to *"No"*
      2. Assign the value of **visit_detail_id** to the column **parent_visit_detail** .
   3. Check if they are partially contained (`identify_partial_rows()`):
      1. If these conditions are verified:
         - **main_visit** -> *"Unknown"*
         - **visit_detail_start_datetime** < **"visit_end_datetime"**
         - **visit_detail_end_datetime** > **"visit_end_datetime"**
      2. Promote **is_partial** value to *True*
   4. For contained visits do (`update_partial_rows()`):
      1. Promote **main_visit** value to *"No"*
      2. Assign to **visit_end_datetime** the value of the latest **visit_detail_end_datetime**.
   5. Check if they are not contained (`identify_not_contained_rows()`):
      1. If these conditions are verified:
         - **main_visit** -> *"Unknown"*
         - **visit_detail_start_datetime** >= **"visit_end_datetime"**
      2. Promote **not_contained** value to *True*
   6. For not contained visits do (`update_not_contained_rows()`):
      - These records have to be reanalyzed. They are **main_visit** candidates.
      1. For not_contained records, we group by patient and extract new values for **visit_detail_id**, **visit_detail_start_datetime** and **visit_detail_end_datetime**  from the earliest *Unknown* record.
      2. For these not contained records, replace values columns **visit_start_datetime**, **visit_end_datetime** and **visit_detail_id_original** with the newest candidates.
5. Go back to step 3. Repeat until there are no more **main_visit** == *"Unknown"* columns or a safe iteration threshold is surpassed.
6. Assign a unique visit_occurrence_id to main visits.


#### Detailed step by step test

##### Test dataset creation

We are going to create a dataset with all the issues we can find.

- Posterior visits completely contained in a previous visit.
  * (2020-01-01, 2020-02-01) contains (2020-01-02, 2020-02-02) and (2020-01-04, 2020-02-04) 
    - We want to keep only the first.
- Posterior dates that are partially contained in a previous visit.
  * (2020-03-01, 2020-04-01) partially contains (2020-03-15, 2020-04-15) 
    - Here we want to combine both => (2020-03-01, 2020-04-15). 
    - "Multi-day visits must not overlap, i.e. share days other than start and end days". See [here](https://ohdsi.github.io/CommonDataModel/cdm54.html#visit_occurrence).
- Another issue is when we have several visits partially contained.
  - When we repeat the process, we need to make sure that previous main visits keep the end_dates
  * (2020-06-01, 2020-07-01) partially contains (2020-06-10, 2020-07-10) and (2020-06-20, 2020-07-20)
- Another issue is when a posterior start_date lies after the first candidate end_date but will be included when a later partially contained visit extends the original end_Date
  - (2020-08-01, 2020-09-01) partially contains (2020-08-10, 2020-09-10). (2020-09-02, 2020-09-20) is not contained at all by the first visit, but is partially contained when the second visit joins the first.
- If we have three visits that form a chain where the previous visit ends exactly when the next visit starts, they are all considered different visits. 
  - This is relevant as the current code can be easily changed to include this case as a partial visit and joined any two visits that share start-end. However, if a third visit was chained to the second, this would be identified as an independent visit, causing an inconsistency in which visits are considered contained or not.
  - This could be solved (theoretically, not tested), by iterating until all partial visits are resolved (steps 4.1 to 4.4) before checking for not contained visits (steps 4.5 and 4.6). This is not done because it is not neccesary for the current implementation.
- Make sure we do not mix visits from different patients.

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import warnings

column_names = [
    "person_id",
    "start_date",
    "end_date",
    "type_concept",
    "visit_concept_id",
    "provider_id",
]
filas = [
    # == Problema de fechas ==
    # We want two main visits, so we know we can distinguish between them
    # Each main visit will have a completely contained visit and a partially contained visit
    # Each main visit will have two type_concept codes
    # -- Main visit 1 --
    # Normal application works?
    (1, "2020-01-01", "2020-02-01", 1, 9202, 0),
    # Completely contained, same type_concept
    (1, "2020-01-02", "2020-01-02", 1, 9202, 0),
    # Completely contained, different type_concept
    (1, "2020-01-04", "2020-01-04", 2, 9202, 0),
    # Partially contained, same type_concept
    (1, "2020-01-06", "2020-02-06", 1, 9202, 0),
    # Partially contained, different type_concept
    (1, "2020-01-08", "2020-02-08", 2, 9202, 0),
    #
    # -- Main visit 2 --
    # Works with second iteration?
    (1, "2020-03-01", "2020-04-01", 1, 9202, 0),
    # Completely contained, same type_concept
    (1, "2020-03-02", "2020-03-02", 1, 9202, 0),
    # Completely contained, different type_concept
    (1, "2020-03-04", "2020-03-04", 2, 9202, 0),
    # Partially contained, same type_concept
    (1, "2020-03-06", "2020-04-06", 1, 9202, 0),
    # Partially contained, different type_concept
    (1, "2020-03-08", "2020-04-08", 2, 9202, 0),
    #
    # -- Main visit 3 --
    # Latest end_date is not the same start_date as main_visit?
    (1, "2020-06-01", "2020-07-01", 1, 9202, 0),
    # First partially contained visit is not the latest end_date
    (1, "2020-06-10", "2020-07-10", 1, 9202, 0),
    (1, "2020-06-20", "2020-07-20", 1, 9202, 0),
    #
    # -- Main visit 4 --
    # Third date is contained in the combination of first and second visit
    (1, "2020-08-01", "2020-09-01", 1, 9202, 0),
    (1, "2020-08-10", "2020-09-10", 1, 9202, 0),
    (1, "2020-09-02", "2020-09-20", 1, 9202, 0),
    # == Problema de person_id ==
    # Dos personas distintas
    (2, "2021-01-01", "2021-01-01", 1, 9202, 0),
    (2, "2021-02-01", "2021-02-01", 1, 9202, 0),
    # Comparte fecha con 1
    (3, "2021-02-01", "2021-02-01", 2, 9202, 0),
    # No comparte fecha con 1
    (3, "2021-03-01", "2021-03-01", 2, 9202, 0),
    #
    # == Problema de type_concept ==
    (4, "2022-03-01", "2022-04-01", 1, 9202, 0),
    # Misma persona y fecha, sólo debe quedar type_concept == 1
    (4, "2022-03-01", "2022-04-01", 2, 9202, 0),
    # == Problema con días exactos ==
    # Cuando hay tres citas y el fin de la segunda coincide con
    # el principio de la tercera, se queda en dos registros separados
    (5, "2020-01-01 01:00", "2020-02-01 01:00", 1, 9202, 0),
    (5, "2020-02-01 01:00", "2020-03-01 01:00", 2, 9202, 0),
    (5, "2020-03-01 01:00", "2020-04-01 01:00", 2, 9202, 0),
]
df_raw = pd.DataFrame.from_records(filas, columns=column_names)
df_raw["start_date"] = pd.to_datetime(df_raw["start_date"], format="ISO8601")
df_raw["end_date"] = pd.to_datetime(df_raw["end_date"], format="ISO8601")
(print(df_raw.info()))
df_raw = pl.DataFrame(df_raw)
df_raw

##### 1. Build VISIT_DETAIL

1. Build the frame for the VISIT_DETAIL table (`build_visit_detail()`).
   1. Sort by `person_id`(asc), `start_date` (asc), `end_date` (desc) and `type_concept` (asc).
      * This way, we make sure that, for each `start_date`, the first column will have the latest `end_date`. This latest `end_date` is the visit that might contain other visits with the same `start_date`.
      * The `type_concept` column should be transformed first to a category datatype with a predefined order. This way the sorting will take that into account as well.
   2. Generate the field `visit_detail_id` with every visit.
   3. Rename columns to fit the VISIT_DETAIL structure.

In [ ]:
def build_visit_detail(df):
    return (
        # First we do the sorting
        df.sort(
            ["person_id", "start_date", "end_date", "type_concept"],
            descending=[False, False, True, False],
        )
        # Assign the visit_detail_id
        .with_columns(visit_detail_id=pl.int_range(pl.len()))
        # Rename columns
        .rename(
            {
                "start_date": "visit_detail_start_datetime",
                "end_date": "visit_detail_end_datetime",
                "type_concept": "visit_detail_type_concept_id",
            }
        )
    )


visit_detail = build_visit_detail(df_raw)
visit_detail

##### 2. Extend VISIT_DETAIL

2. Extend the VISIT_DETAIL table to prepare for VISIT_OCCURRENCE generation (`build_visit_detail_extended()`):
   1. Generate placeholders for future VISIT_OCCURRENCE columns and for temporary columns.
      1. Group by each patient, generating:
        - **visit_start_datetime**: Initially the first **visit_detail_start_datetime**. It will be the **visit_start_datetime** VISIT_OCCCURRENCE column.
        - **visit_end_datetime**: Initially the first **visit_detail_end_datetime**. It will be the **visit_start_datetime** VISIT_OCCCURRENCE column. 
          - If visits needs to be joined together, this record will have the latest joined visit.
        - **visit_detail_id_original**: Initially the first **visit_detail_id**. It will be the **visit_detail_id** VISIT_OCCCURRENCE column.
        - **main_visit**: Initially will have a constant *"Unknown"* value for every record. It will be updated to identify main visits (Changed to "Yes") and the rest (Changed to "No").
      2. Join with VISIT_DETAIL to get the first row (earliest start_date, latest end_date) for each patient.
   2. Generate the *main_visit* column.
      - This field has 3 possibilities: 
         - *Yes:* It was checked and visit is a **main_visit**.
         - *No:* It was checked and visit is NOT a **main_visit**.
         - *Unknown:* We do not know if it is **main_visit** yet.
       - We create this column as a category. Initially all records are *Unknown*. 
   3. Generate the auxiliary columns:
      - `is_contained`, to mark completely contained records.
      - `is_partial`, to mark partialy contained records.
      - `is_not_contained`, to marked not contained records, which could be main visits.
      - `parent_visit_detail_id`, to record the associated **main_visit** when it is found.

In [ ]:
def build_visit_detail_extended(visit_detail):
    # Get the first date of every person
    visit_occurrence_dates = visit_detail.group_by(
        "person_id", maintain_order=True
    ).agg(
        visit_start_datetime=pl.col("visit_detail_start_datetime").first(),
        visit_end_datetime=pl.col("visit_detail_end_datetime").first(),
        visit_detail_id_original=pl.col("visit_detail_id").first(),
    )

    # Join and return
    return visit_detail.join(
        visit_occurrence_dates, on="person_id", how="left"
    ).with_columns(
        main_visit=pl.lit("Unknown").cast(pl.Enum(["Yes", "No", "Unknown"])),
        is_contained=pl.lit(False),
        is_partial=pl.lit(False),
        not_contained=pl.lit(False),
        parent_visit_detail_id=pl.lit(None),
    )


df = build_visit_detail_extended(visit_detail)
df

##### 3. Identify main_visit

3. Identify the next batch of **main_visit** (`identify_next_main_visits()`):
   1. They are the most recent that verify:
      1. They are *"Unknown"*
      2. **visit_detail_id** == **visit_detail_id_original**
   2. Promote **main_visit** value to *"Yes"*

In [ ]:
# Identify main_visits
def identify_next_main_visits(df):
    return df.with_columns(
        # Set new possible main visit as "Yes"
        # This is the first visit that checks visit_detail_id_original == visit_detail_id
        # This is because the when start the column, we use the first visit_detail_id for
        # all records of the patient
        main_visit=(
            pl.when(
                (pl.col("main_visit") == "Unknown")
                & (pl.col("visit_detail_id_original") == pl.col("visit_detail_id"))
            )
            .then(pl.lit("Yes"))
            .otherwise(pl.col("main_visit"))
        )
    ).with_columns(
        # Reset flags
        is_contained=pl.when(pl.col("main_visit").is_in(["Unknown", "Yes"]))
        .then(pl.lit(False))
        .otherwise(pl.col("is_contained")),
        is_partial=pl.when(pl.col("main_visit").is_in(["Unknown", "Yes"]))
        .then(pl.lit(False))
        .otherwise(pl.col("is_partial")),
        not_contained=pl.when(pl.col("main_visit").is_in(["Unknown", "Yes"]))
        .then(pl.lit(False))
        .otherwise(pl.col("not_contained")),
    )


df = identify_next_main_visits(df)
df

##### 4.1 Check visits - Contained

   1. Check if they are completely contained (`identify_contained_rows()`):
      1. If these conditions are verified:
         - **main_visit** -> *"Unknown"*
         - **visit_start_datetime** <= **"visit_detail_start_datetime"**
         - **visit_end_datetime** >= **"visit_detail_end_datetime"**
      2. Promote **is_contained** value to *True*
   2. For contained visits
      1. Promote **main_visit** value to *"No"*
      2. Assign the value of **visit_detail_id** to the column **parent_visit_detail** (`update_contained_rows()`).

In [ ]:
def identify_contained_rows(df):
    return df.with_columns(
        is_contained=pl.when(
            (pl.col("main_visit") == "Unknown")
            & (pl.col("visit_start_datetime") <= pl.col("visit_detail_start_datetime"))
            & (pl.col("visit_end_datetime") >= pl.col("visit_detail_end_datetime"))
        )
        .then(True)
        .otherwise(pl.col("is_contained"))
    )


df = identify_contained_rows(df)
df

In [ ]:
def update_contained_rows(df):

    return df.with_columns(
        # Update main_visit to mark contained visits
        main_visit=(
            pl.when((pl.col("is_contained") == True))
            .then(pl.lit("No"))
            .otherwise(pl.col("main_visit"))
        ),
        # Build the parent_visit_detail_id, since we are here
        parent_visit_detail_id=(
            pl.when(
                (pl.col("is_contained") == True)
                & (pl.col("visit_detail_id") != pl.col("visit_detail_id_original"))
            )
            .then(pl.col("visit_detail_id_original"))
            .otherwise(pl.col("parent_visit_detail_id"))
            .cast(pl.Int32())
        ),
    )


df = update_contained_rows(df)
df

##### 4.2 Check visits - Partial

   3. Check if they are partially contained (`identify_partial_rows()`):
      1. If these conditions are verified:
         - **main_visit** -> *"Unknown"*
         - **visit_detail_start_datetime** < **"visit_end_datetime"**
         - **visit_detail_end_datetime** > **"visit_end_datetime"**
      2. Promote **is_partial** value to *True*
   4. For contained visits do (`update_partial_rows()`):
      1. Promote **main_visit** value to *"No"*
      2. Assign to **visit_end_datetime** the value of the latest **visit_detail_end_datetime**.


In [ ]:
def identify_partial_rows(df):
    return df.with_columns(
        is_partial=pl.when(
            (pl.col("main_visit") == "Unknown")
            & (pl.col("visit_detail_start_datetime") < pl.col("visit_end_datetime"))
            & (pl.col("visit_detail_end_datetime") > pl.col("visit_end_datetime"))
        )
        .then(True)
        .otherwise(pl.col("is_partial"))
    )


df = identify_partial_rows(df)
df

In [ ]:
def update_partial_rows(df):

    latest_date = (
        df.filter(pl.col("is_partial") == True)
        .group_by(["person_id", "visit_detail_id_original"], maintain_order=True)
        .agg(latest_end_datetime=pl.col("visit_detail_end_datetime").max())
    )
    return (
        # Join back to the main dataframe and update visit_end_datetime
        df.join(
            latest_date,
            on=["person_id", "visit_detail_id_original"],
            how="left",
        )
        .with_columns(
            main_visit=(
                pl.when((pl.col("is_partial") == True))
                .then(pl.lit("No"))
                .otherwise(pl.col("main_visit"))
            ),
            visit_end_datetime=pl.when(
                pl.col("visit_end_datetime") != pl.col("latest_end_datetime")
            )
            .then(
                pl.coalesce(
                    [pl.col("latest_end_datetime"), pl.col("visit_detail_end_datetime")]
                )
            )
            .otherwise(pl.col("visit_end_datetime")),
        )
        .drop("latest_end_datetime")
    )


df = update_partial_rows(df)
df

##### 4.3 Check visits - Not Contained

   5. Check if they are not contained (`identify_not_contained_rows()`):
      1. If these conditions are verified:
         - **main_visit** -> *"Unknown"*
         - **visit_detail_start_datetime** >= **"visit_end_datetime"**
      2. Promote **not_contained** value to *True*
   6. For not contained visits do (`update_not_contained_rows()`):
      - These records have to be reanalyzed. They are **main_visit** candidates.
      1. For not_contained records, we group by patient and extract new values for **visit_detail_id**, **visit_detail_start_datetime** and **visit_detail_end_datetime**  from the earliest *Unknown* record.
      2. For these not contained records, replace values columns **visit_start_datetime**, **visit_end_datetime** and **visit_detail_id_original** with the newest candidates.


In [ ]:
def identify_not_contained_rows(df):
    return df.with_columns(
        not_contained=pl.when(
            (pl.col("main_visit") == "Unknown")
            & (pl.col("visit_detail_start_datetime") >= pl.col("visit_end_datetime"))
        )
        .then(True)
        .otherwise(pl.col("not_contained"))
    )


df = identify_not_contained_rows(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

In [ ]:
def update_not_contained_rows(df):

    newest_not_contained = (
        df.filter(pl.col("not_contained") == True)
        .group_by("person_id", maintain_order=True)
        .agg(
            visit_detail_id_newest=pl.col("visit_detail_id").first(),
            visit_detail_start_datetime_newest=pl.col(
                "visit_detail_start_datetime"
            ).first(),
            visit_detail_end_datetime_newest=pl.col(
                "visit_detail_end_datetime"
            ).first(),
        )
    )

    return (
        # Join back to the main dataframe and update visit_end_datetime
        df.join(
            newest_not_contained,
            on="person_id",
            how="left",
        )
        # Update the values on visit_detail_id_original, visit_start_datetime and visit_end_datetime
        .with_columns(
            visit_detail_id_original=pl.when((pl.col("not_contained") == True))
            .then(pl.col("visit_detail_id_newest"))
            .otherwise(pl.col("visit_detail_id_original")),
            visit_start_datetime=pl.when((pl.col("not_contained") == True))
            .then(pl.col("visit_detail_start_datetime_newest"))
            .otherwise(pl.col("visit_start_datetime")),
            visit_end_datetime=pl.when((pl.col("not_contained") == True))
            .then(pl.col("visit_detail_end_datetime_newest"))
            .otherwise(pl.col("visit_end_datetime")),
        ).drop(
            pl.col(
                "visit_detail_id_newest",
                "visit_detail_start_datetime_newest",
                "visit_detail_end_datetime_newest",
            )
        )
    )


df = update_not_contained_rows(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

##### 5. Repeat from step 3


5. Go back to step 3. Repeat until there are no more **main_visit** == *"Unknown"* columns or a safe iteration threshold is surpassed.


Re-identify main_visits and reset flags

In [ ]:
# == Second lap ==
# Re-identify main_visits
df = identify_next_main_visits(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

In [ ]:
# Update completely contained rows
df = identify_contained_rows(df)
df = update_contained_rows(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

In [ ]:
# Update partially contained rows
df = identify_partial_rows(df)
df = update_partial_rows(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

In [ ]:
# Update not contained rows
df = identify_not_contained_rows(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

In [ ]:
df = update_not_contained_rows(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

In [ ]:
# == Third lap ==
df = identify_next_main_visits(df)
# Update completely contained rows
df = identify_contained_rows(df)
df = update_contained_rows(df)
# Update partially contained rows
df = identify_partial_rows(df)
df = update_partial_rows(df)
# Update not contained rows
df = identify_not_contained_rows(df)
df = update_not_contained_rows(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

In [ ]:
# == Fourth lap ==
df = identify_next_main_visits(df)
# Update completely contained rows
df = identify_contained_rows(df)
df = update_contained_rows(df)
# Update partially contained rows
df = identify_partial_rows(df)
df = update_partial_rows(df)
# Update not contained rows
df = identify_not_contained_rows(df)
df = update_not_contained_rows(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

In [ ]:
# == Fifth lap ==
df = identify_next_main_visits(df)
# Update completely contained rows
df = identify_contained_rows(df)
df = update_contained_rows(df)
# Update partially contained rows
df = identify_partial_rows(df)
df = update_partial_rows(df)
# Update not contained rows
df = identify_not_contained_rows(df)
df = update_not_contained_rows(df)
df.drop(["visit_detail_type_concept_id", "visit_concept_id", "provider_id"])

No Unkowns left!

##### 6. Assign unique visit_occurrence_id

6. Assign a unique visit_occurrence_id to main visits.

In [ ]:
def assign_visit_occurrence_id(visit_occurrence):

    return (
        # Create a helper column to track main visit sequence
        visit_occurrence.with_columns(is_main_visit=(pl.col("main_visit") == "Yes"))
        # Assign a unique identifier only to main visits using row_number and clean the helper
        .with_columns(
            visit_occurrence_id=pl.when(pl.col("is_main_visit"))
            .then(pl.col("is_main_visit").cast(pl.Int32).cum_sum() - 1)
            .otherwise(None)
        ).drop("is_main_visit")
        # Fill the rest using forward fill (ffill)
        .with_columns(visit_occurrence_id=pl.col("visit_occurrence_id").forward_fill())
    )


visit_occurrence = assign_visit_occurrence_id(df)
visit_occurrence

We need to build the VISIT_DETAIL and VISIT_OCCURRENCE table by removing all the helper columns we have created

In [ ]:
def build_visit_occurrence(df, verbose=0, n_iter_max=1000):
    # -- Initialization --
    # Get the core of the visit_detail table
    df = build_visit_detail(df)
    # Extend the table for processing
    df = build_visit_detail_extended(df)
    # Initialize counters for the while loop
    n_unknown = df.filter(pl.col("main_visit") == "Unknown").select(pl.len()).item()
    n_iter = 0

    # Look for next batch of main_visits
    df = identify_next_main_visits(df)

    # -- Loop --
    while n_unknown > 0 and n_iter < n_iter_max:
        if verbose > 0:
            print(f"Iter {n_iter:>2}: {n_unknown} unknown rows left.")

        # Identify and update completely contained visits
        df = identify_contained_rows(df)
        df = update_contained_rows(df)

        # Identify and update partially contained visits
        df = identify_partial_rows(df)
        df = update_partial_rows(df)

        # Identify and update not contained visits
        df = identify_not_contained_rows(df)
        df = update_not_contained_rows(df)

        if verbose > 1:
            print(df.filter(pl.col("main_visit") == "Unknown").head(5))

        # Look for next batch of main_visits
        df = identify_next_main_visits(df)

        # Update conditions
        n_unknown = (
            df.filter((pl.col("main_visit") == "Unknown")).select(pl.len()).item()
        )
        n_iter += 1
        if n_iter == n_iter_max and n_unknown > 0:
            warnings.warn(
                f"{n_unknown} rows still unresolved after {n_iter_max} iterations."
            )

    # Assign an unique visit_occurrence_id only to main_visits
    df = assign_visit_occurrence_id(df)

    # Drop the extra helper columns
    df = df.drop(
        # Drop helpers
        pl.col("visit_detail_id_original"),
        pl.col("is_contained"),
        pl.col("is_partial"),
        pl.col("not_contained"),
    )

    # -- Build the core of the visit_detal table --
    # Drop visit_occurrence columns
    visit_detail = df.drop(
        pl.col("visit_start_datetime"),
        pl.col("visit_end_datetime"),
        pl.col(
            "main_visit"
        ),  # This one is dropped here so it can be used for visit_occurrence
    )

    # -- Build the core of the visit_occurrence table --
    # Get only main visits
    visit_occurrence = df.filter(pl.col("main_visit") == "Yes").drop(
        pl.col("main_visit")
    )

    # Rename columns
    visit_occurrence = visit_occurrence.rename(
        {
            "visit_detail_type_concept_id": "visit_type_concept_id",
        }
    )

    # Drop columns from visit_detail
    visit_occurrence = visit_occurrence.drop(
        pl.col("visit_detail_start_datetime"),
        pl.col("visit_detail_end_datetime"),
        pl.col("visit_detail_id"),
        pl.col("parent_visit_detail_id"),
    )

    # n_unknown = df.filter(pl.col("main_visit") == "Unknown").select(pl.len()).collect().item() # Lazy
    n_unknown = (
        df.filter(pl.col("main_visit") == "Unknown").select(pl.len()).item()
    )  # Not Lazy

    # return visit_detail.collect(), visit_occurrence.collect()
    return visit_detail, visit_occurrence


# visit_detail, visit_occurrence = build_visit_occurrence(df_raw_pl, verbose=2,n_iter_max=7)
visit_detail, visit_occurrence = build_visit_occurrence(
    df_raw, verbose=1, n_iter_max=10
)

In [ ]:
visit_detail

In [ ]:
visit_occurrence

In [ ]:
# Check that visit_detail has the correct visit_occurrence_id asssignation
assert (
    np.all(
        visit_detail.select("visit_occurrence_id")
        == pl.Series(
            [
                0,
                0,
                0,
                0,
                0,
                1,
                1,
                1,
                1,
                1,
                2,
                2,
                2,
                3,
                3,
                3,
                4,
                5,
                6,
                7,
                8,
                8,
                9,
                10,
                11,
            ]
        )
    )
    == True
)

In [ ]:
# Check that visit_occurrence has the correct shape
assert visit_occurrence.shape[0] == 12

In [ ]:
# Check that visit_occurrence has the correct people
assert (
    np.all(
        visit_occurrence.select("person_id")
        == pl.Series([1, 1, 1, 1, 2, 2, 3, 3, 4, 5, 5, 5])
    )
    == True
)

In [ ]:
# Check that visit_occurrence has the same start dates
assert (
    pl.Series(
        [
            "2020-01-01 00:00:00",
            "2020-03-01 00:00:00",
            "2020-06-01 00:00:00",
            "2020-08-01 00:00:00",
            "2021-01-01 00:00:00",
            "2021-02-01 00:00:00",
            "2021-02-01 00:00:00",
            "2021-03-01 00:00:00",
            "2022-03-01 00:00:00",
            "2020-01-01 01:00:00",
            "2020-02-01 01:00:00",
            "2020-03-01 01:00:00",
        ]
    ).str.to_datetime()
    == visit_occurrence["visit_start_datetime"]
).all()
# Check that visit_occurrence has the same start dates
assert (
    pl.Series(
        [
            "2020-02-08 00:00:00",
            "2020-04-08 00:00:00",
            "2020-07-20 00:00:00",
            "2020-09-20 00:00:00",
            "2021-01-01 00:00:00",
            "2021-02-01 00:00:00",
            "2021-02-01 00:00:00",
            "2021-03-01 00:00:00",
            "2022-04-01 00:00:00",
            "2020-02-01 01:00:00",
            "2020-03-01 01:00:00",
            "2020-04-01 01:00:00",
        ]
    ).str.to_datetime()
    == visit_occurrence["visit_end_datetime"]
).all()

In [ ]:
%timeit -n 10 -r 5 build_visit_occurrence(df_raw)

## 1.3 Test with big datasets

Vamos a comparar la velocidad de ambos métodos con datasets grandes.

Nos traemos la función para generar datasets

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
from pyarrow import parquet


def create_sample_df(
    n: int = 1000,
    n_dates: int = 50,
    first_date: str = "2020-01-01",
    last_date: str = "2023-01-01",
    mean_duration_days: int = 60,
    std_duration_days: int = 180,
) -> pd.DataFrame:

    # == Parameters ==
    np.random.seed(42)
    pd.options.mode.string_storage = "pyarrow"
    # Start date from which to start the dates
    first_date = pd.to_datetime(first_date)
    last_date = pd.to_datetime(last_date)
    max_days = (last_date - first_date).days

    # == Generate IDs randomly ==
    # -- Generate the Ids
    people = np.random.randint(10000000, 99999999 + 1, size=n)
    person_id = np.random.choice(people, n * n_dates)

    # == Generate random dates ==
    # Generate random integers for days and convert to timedelta
    random_days = np.random.randint(0, max_days, size=n * n_dates)
    # Create the columns
    observation_start_date = first_date + pd.to_timedelta(random_days, unit="D")
    # Generate a gaussian sample of dates
    random_days = np.random.normal(
        mean_duration_days, std_duration_days, size=n * n_dates
    )
    random_days = np.int32(random_days)
    observation_end_date = observation_start_date + pd.to_timedelta(
        random_days, unit="D"
    )
    # Correct end_dates
    # => If they are smaller than start_date, take start_date
    observation_end_date = np.where(
        observation_end_date < observation_start_date,
        observation_start_date,
        observation_end_date,
    )

    # == Generate the code ==
    period_type_concept_id = np.random.randint(1, 11, size=n * n_dates)

    # == Generate the dataframe ==
    df_raw = {
        "person_id": person_id,
        "observation_period_start_date": observation_start_date,
        "observation_period_end_date": observation_end_date,
        "period_type_concept_id": period_type_concept_id,
    }
    return pd.DataFrame(df_raw)

### Time test

In [ ]:
# Cargamos los datos
df_raw = create_sample_df(n=100000)
df_raw.columns = ["person_id", "start_date", "end_date", "type_concept"]
df_raw.loc[:, "visit_concept_id"] = 9202

df_raw_pl = pl.DataFrame(df_raw)
df_raw_pl

In [ ]:
visit_detail, visit_occurrence = build_visit_occurrence(df_raw_pl, verbose=1)

In [ ]:
visit_detail

In [ ]:
visit_occurrence

In [ ]:
print('\npolars test:')
%timeit -n 10 -r 5 build_visit_occurrence(df_raw_pl, verbose=0)

For n = 1000

    - 7.72 s ± 111 ms per loop (mean ± std. dev. of 5 runs, 10 loops each)
